# IOAI — 2024 Residential Camp Nlp Ner Dutch (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import nltk; nltk.download('conll2002')
from nltk.corpus import conll2002
import json
sents = lambda tag: [[(w, t) for (w, p, t) in s] for s in conll2002.iob_sents(tag)]
json.dump(sents('ned.train'), open('train.json', 'w'))
json.dump([[w for w, t in s] for s in sents('ned.testb')], open('test.json', 'w'))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Dutch Named Entity Recognition (CoNLL-2002)

> **Singapore IOAI 2024 — Residential Camp (NLP)**

Tag every token with a BIO label over 4 entity types (PER/LOC/ORG/MISC) on the Dutch CoNLL-2002 corpus. Score = **macro-F1 over entity tags** (excluding `O`). `train.json` is `[[(token,tag),…],…]`; `test.json` is tokens only; **Submit** grades `submission.csv` (`id,tag`, id=`<sent>_<tok>`). Baseline: per-token features + LinearSVC (the original used a BiLSTM).

In [ ]:
import json
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

train = json.load(open('train.json'))   # [[[token, tag], ...], ...]
test = json.load(open('test.json'))     # [[token, ...], ...]
len(train), len(test)

## Per-token features (word, affixes, shape, neighbours)

In [ ]:
def feats(tokens, i):
    w = tokens[i]
    f = {'w': w.lower(), 'suf3': w[-3:].lower(), 'suf2': w[-2:].lower(), 'pre2': w[:2].lower(),
         'cap': w[:1].isupper(), 'upper': w.isupper(), 'digit': w.isdigit()}
    f['prev'] = tokens[i - 1].lower() if i > 0 else '<s>'
    f['next'] = tokens[i + 1].lower() if i < len(tokens) - 1 else '</s>'
    return f

def build(sents, labeled):
    X, y = [], []
    for s in sents:
        toks = [w for w, t in s] if labeled else s
        for i in range(len(toks)):
            X.append(feats(toks, i))
            if labeled:
                y.append(s[i][1])
    return (X, y) if labeled else X

# self-check: last 10% of train sentences as validation
cut = int(len(train) * 0.9)
Xtr, ytr = build(train[:cut], True)
Xva, yva = build(train[cut:], True)
vec = DictVectorizer(sparse=True)
clf = LinearSVC(C=0.5)
clf.fit(vec.fit_transform(Xtr), ytr)
pva = clf.predict(vec.transform(Xva))
ent = sorted(set(t for t in yva if t != 'O'))
print(f'val macro-F1 (entity tags): {f1_score(yva, pva, labels=ent, average="macro"):.4f}')

## Train on all data, tag the test set, write `submission.csv`

In [ ]:
Xall, yall = build(train, True)
vec = DictVectorizer(sparse=True)
clf = LinearSVC(C=0.5)
clf.fit(vec.fit_transform(Xall), yall)

ids, feat_rows = [], []
for si, toks in enumerate(test):
    for ti in range(len(toks)):
        ids.append(f'{si}_{ti}'); feat_rows.append(feats(toks, ti))
pred = clf.predict(vec.transform(feat_rows))
pd.DataFrame({'id': ids, 'tag': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(ids))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)